# Holt-Winters Grid Search

## Purpose
Conducts an exhaustive grid search over Holt-Winters `(trend, damped_trend, seasonal)` configuration combinations to identify the configuration that minimises walk-forward RMSE on the training dataset.

Invalid combinations are excluded before fitting: `damped_trend=True` requires a non-`None` trend component, and multiplicative components are undefined when any data value is non-positive. A fit validity guard is applied after every `model.fit()` call — smoothing parameters outside `[0, 1]` or non-finite forecasts raise a `RuntimeError`, causing that configuration to be skipped cleanly.

## Inputs
- `data/dataset.csv` — Training dataset (93 monthly observations)

## Outputs
- The RMSE for each valid configuration and the overall best configuration.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from math import sqrt
import itertools
import warnings
warnings.filterwarnings('ignore')

## Load Training Data

In [2]:
series = pd.read_csv('data/dataset.csv', index_col=0, parse_dates=True).iloc[:, 0]
series.index.freq = pd.infer_freq(series.index)
series.head()

Month
1964-01-01    2815
1964-02-01    2672
1964-03-01    2755
1964-04-01    2721
1964-05-01    2946
Freq: MS, Name: Sales, dtype: int64

## Fit Validity Guard

`ExponentialSmoothing` does not expose a convergence flag like ARIMA's `mle_retvals`. The equivalent check is to confirm all smoothing parameters are finite and within `[0, 1]`. This function is called after every fit inside the walk-forward loop; a degenerate result raises a `RuntimeError` so the outer grid search skips that configuration via `except: continue`.

In [3]:
def is_hw_fit_valid(fit):
    """Return True if all smoothing params are finite and within [0, 1].

    ExponentialSmoothing does not expose mle_retvals like ARIMA; checking
    that the estimated smoothing weights are bounded is the equivalent
    empirical guard against a failed/degenerate optimisation.

    Bounds are closed [0, 1]: a value of exactly 0 for smoothing_trend is
    valid — it means the optimiser found no per-step trend update is needed
    (trend stays at its initial estimate). Rejecting 0.0 with a strict
    lower bound would incorrectly flag this legitimate outcome as a failure.
    The true failure signals are NaN, inf, or values outside [0, 1].
    """
    param_keys = ['smoothing_level', 'smoothing_trend', 'smoothing_seasonal']
    for key in param_keys:
        val = fit.params.get(key)
        if val is None:
            continue
        if not np.isfinite(val) or not (0.0 <= val <= 1.0):
            return False
    return True

## Single-Configuration Evaluation Function

Runs walk-forward validation for one `(trend, damped_trend, seasonal, seasonal_periods)` combination and returns the RMSE. Raises `RuntimeError` if any step produces invalid smoothing parameters or a non-finite forecast, allowing the calling grid search to skip the configuration cleanly.

In [4]:
def evaluate_hw_model(X, trend, damped_trend, seasonal, seasonal_periods):
    """Walk-forward validation for one Holt-Winters configuration.
    
    Raises RuntimeError if any step produces invalid smoothing parameters.
    """
    X = X.astype('float64')
    train_size = int(len(X) * 0.50)
    train, test = X[:train_size], X[train_size:]
    history = list(train)
    predictions = []

    for t in range(len(test)):
        model = ExponentialSmoothing(
            history,
            trend=trend,
            damped_trend=damped_trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods,
            initialization_method='estimated'
        )
        fit = model.fit(optimized=True)

        if not is_hw_fit_valid(fit):
            raise RuntimeError(
                f"Invalid smoothing params at step {t}: {fit.params}"
            )

        yhat = fit.forecast(1)[0]
        if not np.isfinite(yhat):
            raise RuntimeError(f"Non-finite forecast at step {t}")

        predictions.append(yhat)
        history.append(test[t])

    rmse = sqrt(mean_squared_error(test, predictions))
    return rmse

## Grid Search Over All Configurations

Iterates over every combination of the supplied `trend`, `damped`, and `seasonal` options, calls `evaluate_hw_model` for each valid combination, and tracks the configuration with the lowest RMSE. Two classes of invalid combination are excluded up front before any fitting is attempted: damping without a trend component, and multiplicative components on non-positive data.

In [5]:
def evaluate_models(dataset, trend_options, damped_options, seasonal_options, seasonal_periods):
    """Grid search over all valid Holt-Winters configurations.

    Invalid combinations are skipped explicitly before fitting:
      - damped_trend=True requires trend to be non-None.
      - Multiplicative components require strictly positive data.
    """
    data_min = dataset.min()
    best_score, best_cfg = float('inf'), None

    for trend, damped, seasonal in itertools.product(
        trend_options, damped_options, seasonal_options
    ):
        # damping only applies when a trend component exists
        if damped and trend is None:
            continue

        # multiplicative components undefined for non-positive data
        if data_min <= 0 and (trend == 'mul' or seasonal == 'mul'):
            continue

        cfg = (trend, damped, seasonal)
        cfg_label = f"trend={trend}, damped={damped}, seasonal={seasonal}"

        try:
            rmse = evaluate_hw_model(
                dataset, trend, damped, seasonal, seasonal_periods
            )
            if rmse < best_score:
                best_score, best_cfg = rmse, cfg
            print(f'HW({cfg_label}) RMSE={rmse:.3f}')
        except Exception:
            continue

    best_label = (
        f"trend={best_cfg[0]}, damped={best_cfg[1]}, seasonal={best_cfg[2]}"
        if best_cfg is not None else "None"
    )
    print(f'\nBest HW({best_label}) RMSE={best_score:.3f}')

## Run Grid Search

In [6]:
# evaluate parameters
trend_options    = [None, 'add', 'mul']
damped_options   = [True, False]
seasonal_options = [None, 'add', 'mul']
seasonal_periods = 12

evaluate_models(
    series.values, trend_options, damped_options,
    seasonal_options, seasonal_periods
)

HW(trend=add, damped=True, seasonal=add) RMSE=969.634
HW(trend=add, damped=True, seasonal=mul) RMSE=848.318
HW(trend=add, damped=False, seasonal=add) RMSE=966.408
HW(trend=add, damped=False, seasonal=mul) RMSE=840.194
HW(trend=mul, damped=True, seasonal=add) RMSE=1775.586
HW(trend=mul, damped=True, seasonal=mul) RMSE=3065.164
HW(trend=mul, damped=False, seasonal=add) RMSE=1011.154
HW(trend=mul, damped=False, seasonal=mul) RMSE=849.691

Best HW(trend=add, damped=False, seasonal=mul) RMSE=840.194
